# BEE 4750 Mini-Project 2: Dispatching New York’s Grid Under a Carbon

Price

**Names**:

**IDs**:

> **Due Date**
>
> Friday, 11/20/26, 9:00pm

## Overview

In 2013, New York’s grid still burned coal, ran the Indian Point nuclear
plant, and had almost no solar. In this project you will model how that
grid met demand hour by hour, find the price of electricity each hour,
and ask what impact a carbon price would have made: first by changing
which plants run, and then by changing what gets built.

### Instructions

- Students in BEE 4750 may work in groups of 2; students in BEE 5750
  work individually.
- Submit a **report of no more than 4 pages**, not counting figures,
  tables, code, and references, as a PDF to Gradescope. Answer every
  numbered question in the report and label each answer with its number
  (for example, **3.2**), then tag those pages in Gradescope.
- Put your code in an appendix or submit the notebook alongside the
  report. The report should read on its own without it.
- Parts 1–4 use linear programming, shadow prices, JuMP, and
  multi-period dispatch (October 14–28 and the October 26 lab). Part 5
  uses capacity expansion (November 2).

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

In [1]:
using JuMP
using HiGHS
using CSV
using DataFrames
using Statistics
using Plots
using Measures

## The System

We treat New York as a single node (no transmission limits) and model
every hour of 2013.

**Demand** is NYISO’s hourly load for 2013, summed over its eleven
zones. It ranges from about 12 GW overnight to a 34 GW summer peak.

**The existing fleet** is New York’s summer 2013 generating capability,
grouped by type. Each plant type can generate up to its available
capacity in each hour. The thermal types (nuclear, coal, steam, combined
cycle, and combustion turbines) can also change their output by at most
their ramp fraction of capacity from one hour to the next. Hydro, wind,
and solar have no ramp limit.

| Type | Capacity (MW) | Variable cost (\$/MWh) | CO<sub>2</sub> (t/MWh) | Ramp (per hour) | Availability |
|:-----------|-----------:|-----------:|-----------:|-----------:|:-----------|
| Nuclear | 5,411 | 8 | 0 | 0.25 | 88% |
| Hydro | 4,276 | 0 | 0 | – | budget |
| Coal | 1,548 | 24 | 1.00 | 0.70 | 100% |
| Oil and gas steam | 9,576 | 45 | 0.58 | 0.70 | 100% |
| Combined cycle | 8,991 | 30 | 0.43 | 0.70 | 100% |
| Combustion turbine | 4,911 | 40 | 0.55 | 1 | 100% |
| Wind | 1,634 | 0 | 0 | – | profile |
| Solar | 31.5 | 0 | 0 | – | profile |

Table 1: New York’s 2013 generating fleet.

The fleet file has a ninth row, combined cycle with carbon capture, with
no capacity: New York had none in 2013. It generates nothing in Parts
1–4 and becomes an option in Part 5.

**Hydro** can be dispatched in any hour up to its capacity, but its
total generation over the year cannot exceed **24,572 GWh**, New York’s
hydro generation in 2012. Most of that capacity is the New York Power
Authority’s plants on the Niagara and St. Lawrence rivers. At Niagara, a
1950 treaty with Canada sets how much of the river’s flow must go over
the falls rather than through the turbines, which limits the water
available for power. Water used in one hour is not available in another.

**Wind and solar** follow hourly capacity-factor profiles, rescaled so
their annual averages match New York’s observed values: 23.4% for wind
and 16.5% for solar.

**Non-served energy** is allowed but penalized at \$10,000/MWh, so the
problem is always feasible. A good plan should have none.

> **Data and Simplifications**
>
> Capacities are from NYISO’s *2013 Load & Capacity Data* report (the
> “Gold Book”), Table III-3a, summer capability; wind and solar use
> nameplate capacity. Coal, combined cycle, and combustion turbine
> variable costs are roughly what these plants cost to run at 2012–13
> fuel prices, and their emissions are the values used in lecture; the
> others are course assumptions. The wind capacity factor is from the
> Gold Book’s 2012 energy for wind plants in service all year, and the
> solar capacity factor is the Long Island Solar Farm’s in NYISO’s 2024
> Gold Book. The ramp fractions are hourly limits: coal’s and combined
> cycle’s are the rates of the large units in the RTS-GMLC test system
> (about 4 MW per minute on a 350 MW unit), steam is assumed to match
> coal, combustion turbines can reach full output within an hour, and
> nuclear’s is a course assumption.
>
> The model leaves out imports (New York imported roughly 15% of its
> electricity), pumped storage (1,407 MW), about 400 MW of small biomass
> and waste plants, and transmission limits between zones.

For comparison, in 2012 New York generated 139,137 GWh in-state: 29.4%
nuclear, 17.7% hydro, 2.2% wind, and 48.1% fossil, of which coal was
3.1%.

The data, ready to use:

In [1]:
# Statewide hourly demand: the sum of NYISO's eleven load zones [MW]
load_data = DataFrame(CSV.File("data/2013_hourly_load_NY.csv"))
zones = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K"]
demand = zeros(nrow(load_data))
for zone in zones
    demand .+= load_data[:, zone]
end

# Hourly wind and solar capacity factors, rescaled to New York's annual averages
profiles = DataFrame(CSV.File("data/wind_solar_capacity_factors.csv"))
wind_cf = profiles.Wind .* (0.234 / mean(profiles.Wind))
solar_cf = profiles.Solar .* (0.165 / mean(profiles.Solar))

# The existing fleet, as in the table above. Wind and solar availability come
# from the hourly profiles, not the Availability column. RampFraction is missing
# for hydro, wind and solar, which have no ramp limit. The last row, combined
# cycle with carbon capture, has no capacity until Part 5.
fleet = DataFrame(CSV.File("data/nyca_2013_fleet.csv"))

hydro_budget = 24_572_000   # annual hydro energy [MWh]
nse_cost = 10_000           # penalty on non-served energy [$/MWh]

## Problems (Total: 100 Points)

### Part 1: Formulate the Dispatch Problem (15 points)

#### Problem 1.1 (10 points)

Formulate a linear program that decides how much each plant type
generates in every hour of 2013 to meet demand at the lowest total
variable cost. Define your decision variables (with units), objective,
and constraints: meeting demand each hour, each plant’s hourly
availability, ramping limits for the thermal types, and the hydro energy
budget. Explain each constraint in a sentence.

#### Problem 1.2 (5 points)

Now add a carbon tax of $\tau$ dollars per tonne of CO<sub>2</sub>. Show
how it enters your objective. Then, **by hand**, find the tax at which
coal falls behind each of the combined cycle, combustion turbine, and
steam plants in the merit order.

### Part 2: Dispatching New York in 2013 (20 points)

#### Problem 2.1 (10 points)

Solve your model in JuMP with no carbon tax. Report the total variable
cost, annual CO<sub>2</sub> emissions, each plant type’s share of
generation, and any non-served energy. Compare your generation shares
with New York’s actual 2012 shares above, and give two reasons for the
largest differences.

#### Problem 2.2 (10 points)

Plot hourly generation by plant type, stacked, for one week in January
and one week in July. Explain the order in which plant types are used,
and how the model uses hydro given its annual budget.

### Part 3: Prices (15 points)

#### Problem 3.1 (8 points)

The shadow price of each hour’s demand constraint is the cost of meeting
one more MWh of demand in that hour: the marginal price of electricity.
Compute it for every hour and plot the prices sorted from highest to
lowest. Count a plant type as setting the price in the hours whose
price, rounded to the cent, equals its variable cost. Which plant type
sets the price most often, and in how many hours do combustion turbines
or steam plants set it?

#### Problem 3.2 (7 points)

Report the shadow price of the hydro budget and interpret it. Suppose
enough extra water could be diverted through the Niagara turbines to
generate one more MWh over the year: what would that MWh be worth, and
why that value? Be careful with the sign.

### Part 4: A Carbon Tax (25 points)

#### Problem 4.1 (12 points)

Solve the dispatch problem with carbon taxes of \$0, \$20, \$40, \$60,
and \$100 per tonne. Report annual emissions and each fossil plant
type’s share of generation at each tax in a table. Between which taxes
do emissions drop, and do the drops match your answer to 1.2? Why does
coal keep running after it falls behind combined cycle plants?

#### Problem 4.2 (13 points)

Is there any tax that brings emissions below 40 million tonnes per year
by changing only which existing plants run? Identify the lowest
emissions redispatch can reach, and explain what sets that floor. What
happens to the average of the year’s hourly prices as the tax rises, and
who pays it?

### Part 5: Building New Capacity (25 points)

The state can now build six kinds of new capacity: combined cycle,
combustion turbines, wind, solar, hydro, and combined cycle with carbon
capture. New capacity behaves like the existing plants of the same type:
the same variable cost, emissions, and availability, and for the gas
plants the same ramp fraction. Two of the six need more detail:

- **Hydro.** The only sites are existing dams that have no turbines, so
  at most 295 MW can be built. We assume each MW of new hydro generates,
  on average, as much as each MW of existing hydro: each new MW raises
  the annual hydro generating capacity by 24,572,000 MWh ÷ 4,276 MW ≈
  5,746 MWh.
- **Combined cycle with carbon capture.** This is the fleet file’s last
  row. It captures 90% of its CO<sub>2</sub>, so it emits 0.05 t/MWh. It
  costs \$39/MWh to run, more than a combined cycle plant, because it
  burns more gas and pays to move and store the CO<sub>2</sub> it
  captures.

Annualized costs of new capacity are in `data/new_capacity_costs.csv`
(in \$ per MW of capacity per year): `Cost2013` is from 2013 cost
estimates and `CostToday` from 2025 ones.

In [1]:
new_costs = DataFrame(CSV.File("data/new_capacity_costs.csv"))
hydro_build_limit = 295   # most new hydro that can be built [MW]

> **New Capacity: Data and Cautions**
>
> The annualized costs are capital costs from the U.S. Energy
> Information Administration’s 2013 and 2025 reports on new generating
> plants, spread over 30 years at 7%, plus fixed operating costs. The
> 295 MW of hydro is Oak Ridge National Laboratory’s 2012 estimate of
> the capacity that New York’s dams without turbines could support.
> Capture’s variable cost of \$39/MWh includes about \$4.50/MWh to move
> and store the 0.45 t of CO<sub>2</sub> it captures from each MWh, at
> \$10 per tonne.
>
> Keep two things in mind when you interpret your results. First, the
> 2013 cost of capture is an estimate for a kind of plant that had not
> been built at commercial scale. Second, the two reports describe
> different hydro plants, 500 MW in 2013 and 100 MW in 2025, so part of
> the increase in hydro’s cost between them comes from comparing a
> larger plant with a smaller one, not from a change in what the same
> plant costs.

#### Problem 5.1 (8 points)

Extend your model to include build decisions, using 2013 costs. Explain
what changes in the objective and the constraints. With no emissions
policy, what does the model build, and why? Compare the cost of a MWh
from new wind with the cost of the fuel it would replace.

#### Problem 5.2 (8 points)

Cap annual emissions at 40 million tonnes, still with 2013 costs. What
does the model build, what does the system cost, and what is the shadow
price of the cap? Explain what the new plants replace.

#### Problem 5.3 (9 points)

Repeat 5.2’s capped problem with today’s costs. What gets built now, and
what is the shadow price of the same cap? Then remove the cap and
instead impose a carbon tax equal to that shadow price. Compare the two
solutions, and explain the result using what you know about shadow
prices. What does the change from 5.2 tell you about the role of a
carbon price when technology costs fall?

> **Working With JuMP Output**
>
> `value.(x)` returns a JuMP container; use `value.(x).data` or index it
> with `value(x[g, t])` to get plain numbers for plotting. End cells
> that build large sets of variables or constraints with a semicolon to
> suppress long printed output. For a constraint `c`, `shadow_price(c)`
> reports how the objective changes as the constraint is relaxed; check
> its sign against your intuition. `fleet.RampFraction` is `missing` for
> the types with no ramp limit; test for that with `ismissing`.

## References

List any external references consulted, including classmates.